In [1]:
%pip install scikit-learn openpyxl fpdf2 matplotlib seaborn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    cohen_kappa_score, matthews_corrcoef, classification_report
)
from ultralytics import YOLO
import warnings
import os
warnings.filterwarnings('ignore')

# Load model
model_path = r'.\ml_pipeline_Obstacle_det\runs\detect\Drishti_Final_Push\v11n_augmented_853\weights\best.pt'
model = YOLO(model_path)

# Correct paths
val_images = r'.\ml_pipeline_Obstacle_det\drishti_final_split\images\val'
val_labels = r'.\ml_pipeline_Obstacle_det\drishti_final_split\labels\val'

os.makedirs('evaluation_results', exist_ok=True)

print("Model loaded successfully!")
print(f"Classes: {model.names}")
print(f"Total images: {len(list(Path(val_images).glob('*.jpg')))}")

Model loaded successfully!
Classes: {0: 'motorcycle', 1: 'person', 2: 'plant pot', 3: 'pole', 4: 'pothole', 5: 'tree', 6: 'zebra cross'}
Total images: 1290


In [5]:
print(os.path.exists(val_images))
print(os.path.exists(val_labels))
print(os.path.exists(r'.\ml_pipeline_Obstacle_det\drishti_final_split\data_final.yaml'))

True
True
True


In [6]:
print("Collecting predictions from validation set...")

all_scores = {name: [] for name in model.names.values()}
all_labels = {name: [] for name in model.names.values()}
all_true   = []
all_pred   = []

for img_file in Path(val_images).glob('*.jpg'):
    result     = model.predict(img_file, conf=0.01, verbose=False)[0]
    label_file = Path(val_labels) / (img_file.stem + '.txt')

    gt_classes = []
    if label_file.exists():
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    # ✅ Fix: convert float to int safely
                    cls = int(float(parts[0]))
                    gt_classes.append(cls)

    pred_classes = [int(box.cls) for box in result.boxes]

    # For ROC / PR / F1 curves
    for cls_id, cls_name in model.names.items():
        gt    = 1 if cls_id in gt_classes else 0
        preds = [float(box.conf) for box in result.boxes if int(box.cls) == cls_id]
        score = max(preds) if preds else 0.0
        all_scores[cls_name].append(score)
        all_labels[cls_name].append(gt)

    # For confusion matrix
    for gt in gt_classes:
        gt_name   = model.names[gt]
        pred_name = model.names[pred_classes[0]] if pred_classes else 'none'
        all_true.append(gt_name)
        all_pred.append(pred_name)

print(f"Collected {len(all_true)} predictions!")

Collected 1636 predictions!


In [8]:
import os

# Fix yaml file first
correct_path = os.path.abspath(r'.\ml_pipeline_Obstacle_det\drishti_final_split')

yaml_content = f"""path: {correct_path}
train: images/train
val: images/val
test: images/test

names:
  0: motorcycle
  1: person
  2: plant pot
  3: pole
  4: pothole
  5: tree
  6: zebra cross
"""

yaml_path = r'.\ml_pipeline_Obstacle_det\drishti_final_split\data_final.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print("yaml updated!")

# Run official YOLO validation
print("Running official YOLO validation...")
metrics   = model.val(data=yaml_path, split='val', verbose=False)

map50     = metrics.box.map50
map5095   = metrics.box.map
precision = metrics.box.mp
recall    = metrics.box.mr

print("\n" + "="*45)
print("       OFFICIAL YOLO VALIDATION METRICS")
print("="*45)
print(f"  mAP@50:          {map50:.4f}  ({map50*100:.2f}%)")
print(f"  mAP@50-95:       {map5095:.4f}  ({map5095*100:.2f}%)")
print(f"  Precision:       {precision:.4f}  ({precision*100:.2f}%)")
print(f"  Recall:          {recall:.4f}  ({recall*100:.2f}%)")
print("="*45)

yaml updated!
Running official YOLO validation...
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0.dev20260214+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 384.4198.8 MB/s, size: 152.9 KB)
val: Scanning C:\Users\USER\Downloads\Obstacle_Detection_Project\drishti-final\ml_pipeline_Obstacle_det\drishti_final_split\labels\val.cache... 1290 images, 386 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1290/1290  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 81/81 7.6it/s 10.7s0.1s
                   all       1290       1636      0.942      0.919      0.957      0.695
Speed: 1.3ms preprocess, 2.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\USER\Downloads\Obstacle_Detection_Project\drishti-final\runs\detect\val2

       OFFICIAL YOLO VALIDATION METRICS
  mAP@50:          0.9574  (95.74%)
  mAP@50-95:       0.6946  (69.46%)
  Precision:    

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, matthews_corrcoef, classification_report

# 1. Define classes from your YOLO model
class_names = list(model.names.values())

# 2. Filter predictions to ensure they match expected class names
# This prevents errors if 'none' or '0.0' strings were captured
filtered_true = [t for t in all_true if t in class_names]
filtered_pred = [p if p in class_names else class_names[0] for p in all_pred]

# 3. Calculate Core Machine Learning Metrics
accuracy = accuracy_score(filtered_true, filtered_pred)
f1       = f1_score(filtered_true, filtered_pred, average='weighted', zero_division=0)
kappa    = cohen_kappa_score(filtered_true, filtered_pred)
mcc      = matthews_corrcoef(filtered_true, filtered_pred)

# 4. Construct the Report String
report_content = f"""
{"="*45}
         DRISHTI: CLASSIFICATION METRICS
{"="*45}
  Accuracy:        {accuracy:.4f}  ({accuracy*100:.2f}%)
  F1 Score:        {f1:.4f}  ({f1*100:.2f}%)
  Kappa Score:     {kappa:.4f}
  MCC Score:       {mcc:.4f}
{"="*45}

Per-Class Classification Report:
{classification_report(filtered_true, filtered_pred, target_names=class_names, zero_division=0)}
"""

# 5. Print to console for immediate review
print(report_content)

# 6. Securely save to a text file in your Downloads folder
file_name = "drishti_classification_report.txt"
try:
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(report_content)
    print(f"Success: Report securely saved as '{file_name}'")
except Exception as e:
    print(f"Error saving file: {e}")


         DRISHTI: CLASSIFICATION METRICS
  Accuracy:        0.8123  (81.23%)
  F1 Score:        0.8110  (81.10%)
  Kappa Score:     0.7805
  MCC Score:       0.7818

Per-Class Classification Report:
              precision    recall  f1-score   support

  motorcycle       0.65      0.74      0.69       240
      person       0.88      0.92      0.90       280
   plant pot       0.94      0.94      0.94       263
        pole       0.84      0.56      0.67       209
     pothole       0.95      0.92      0.94       217
        tree       0.69      0.82      0.75       203
 zebra cross       0.76      0.73      0.74       224

    accuracy                           0.81      1636
   macro avg       0.82      0.80      0.80      1636
weighted avg       0.82      0.81      0.81      1636


Success: Report securely saved as 'drishti_classification_report.txt'


In [11]:
from sklearn.model_selection import StratifiedKFold

print("Running 5-Fold Cross Validation...")

primary_class = class_names[0]
scores_arr    = np.array(all_scores[primary_class])
labels_arr    = np.array(all_labels[primary_class])

kf           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(scores_arr, labels_arr)):
    val_scores      = scores_arr[val_idx]
    val_labels_fold = labels_arr[val_idx]
    preds           = (val_scores >= 0.5).astype(int)
    acc             = accuracy_score(val_labels_fold, preds)
    f1_val          = f1_score(val_labels_fold, preds, zero_division=0)
    fold_results.append({'Fold': fold+1, 'Accuracy': acc, 'F1 Score': f1_val})
    print(f"  Fold {fold+1}: Accuracy={acc:.4f}  F1={f1_val:.4f}")

df_kfold = pd.DataFrame(fold_results)
print(f"\n  Mean Accuracy: {df_kfold['Accuracy'].mean():.4f} ± {df_kfold['Accuracy'].std():.4f}")
print(f"  Mean F1 Score: {df_kfold['F1 Score'].mean():.4f} ± {df_kfold['F1 Score'].std():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_kfold))
ax.bar(x - 0.2, df_kfold['Accuracy'], 0.35, label='Accuracy', color='steelblue')
ax.bar(x + 0.2, df_kfold['F1 Score'], 0.35, label='F1 Score', color='coral')
ax.set_xlabel('Fold')
ax.set_ylabel('Score')
ax.set_title('5-Fold Cross Validation Results')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/kfold_crossvalidation.png', dpi=150)
plt.show()
print("K-Fold Cross Validation saved!")

Running 5-Fold Cross Validation...
  Fold 1: Accuracy=0.9884  F1=0.9552
  Fold 2: Accuracy=0.9651  F1=0.8657
  Fold 3: Accuracy=0.9729  F1=0.8923
  Fold 4: Accuracy=0.9690  F1=0.8788
  Fold 5: Accuracy=0.9806  F1=0.9254

  Mean Accuracy: 0.9752 ± 0.0093
  Mean F1 Score: 0.9035 ± 0.0365


<Figure size 800x400 with 1 Axes>

K-Fold Cross Validation saved!


In [12]:
colors = plt.cm.Set1(np.linspace(0, 1, len(model.names)))

cm  = confusion_matrix(filtered_true, filtered_pred, labels=class_names)
fig, ax = plt.subplots(figsize=(10, 8))
im  = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im)
ax.set(xticks=np.arange(len(class_names)),
       yticks=np.arange(len(class_names)),
       xticklabels=class_names,
       yticklabels=class_names,
       title='Confusion Matrix - Drishti Obstacle Detection',
       ylabel='True Label',
       xlabel='Predicted Label')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig('evaluation_results/confusion_matrix.png', dpi=150)
plt.show()
print("Confusion Matrix saved!")

<Figure size 1000x800 with 2 Axes>

Confusion Matrix saved!


In [13]:
roc_auc_scores = {}

plt.figure(figsize=(10, 7))
for (cls_name, scores), color in zip(all_scores.items(), colors):
    labels = all_labels[cls_name]
    if len(set(labels)) < 2:
        print(f"Skipping {cls_name}")
        continue
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[cls_name] = roc_auc
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{cls_name} (AUC={roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC-AUC Curve - Drishti Obstacle Detection', fontsize=15)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/roc_auc_curve.png', dpi=150)
plt.show()
print("ROC-AUC Curve saved!")

<Figure size 1000x700 with 1 Axes>

ROC-AUC Curve saved!


In [14]:
pr_scores = {}

plt.figure(figsize=(10, 7))
for (cls_name, scores), color in zip(all_scores.items(), colors):
    labels = all_labels[cls_name]
    if len(set(labels)) < 2:
        continue
    prec, rec, _ = precision_recall_curve(labels, scores)
    pr_auc       = auc(rec, prec)
    pr_scores[cls_name] = pr_auc
    plt.plot(rec, prec, color=color, lw=2, label=f'{cls_name} (AUC={pr_auc:.2f})')

plt.xlabel('Recall', fontsize=13)
plt.ylabel('Precision', fontsize=13)
plt.title('Precision-Recall Curve - Drishti Obstacle Detection', fontsize=15)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/pr_curve.png', dpi=150)
plt.show()
print("PR Curve saved!")

<Figure size 1000x700 with 1 Axes>

PR Curve saved!


In [15]:
thresholds = np.arange(0.01, 1.0, 0.01)
f1_scores  = {}

plt.figure(figsize=(10, 7))
for (cls_name, scores), color in zip(all_scores.items(), colors):
    labels = all_labels[cls_name]
    if len(set(labels)) < 2:
        continue
    f1s     = [f1_score(labels, (np.array(scores) >= t).astype(int), zero_division=0)
               for t in thresholds]
    best_f1 = max(f1s)
    f1_scores[cls_name] = best_f1
    plt.plot(thresholds, f1s, color=color, lw=2,
             label=f'{cls_name} (Best F1={best_f1:.2f})')

plt.xlabel('Confidence Threshold', fontsize=13)
plt.ylabel('F1 Score', fontsize=13)
plt.title('F1-Confidence Curve - Drishti Obstacle Detection', fontsize=15)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/f1_confidence_curve.png', dpi=150)
plt.show()
print("F1-Confidence Curve saved!")

<Figure size 1000x700 with 1 Axes>

F1-Confidence Curve saved!


In [16]:
print("\n" + "="*50)
print("         OVERFITTING CHECK")
print("="*50)
print(f"  CV Mean Accuracy:    {df_kfold['Accuracy'].mean():.4f}")
print(f"  Test Accuracy:       {accuracy:.4f}")
diff = abs(df_kfold['Accuracy'].mean() - accuracy)
print(f"  Difference:          {diff:.4f}")

if diff < 0.05:
    print(" No overfitting detected - Model generalizes well!")
elif diff < 0.10:
    print(" Slight overfitting - acceptable range")
else:
    print(" Overfitting detected - model needs improvement")
print("="*50)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['CV Mean Accuracy', 'Test Accuracy'],
              [df_kfold['Accuracy'].mean(), accuracy],
              color=['steelblue', 'coral'], width=0.4)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Overfitting Check: CV vs Test Accuracy')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.4f}', ha='center', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/overfitting_check.png', dpi=150)
plt.show()
print("Overfitting Check saved!")


         OVERFITTING CHECK
  CV Mean Accuracy:    0.9752
  Test Accuracy:       0.8123
  Difference:          0.1628
 Overfitting detected - model needs improvement


<Figure size 700x400 with 1 Axes>

Overfitting Check saved!


In [18]:
summary = {
    'Metric': [
        'mAP@50', 'mAP@50-95', 'Precision', 'Recall',
        'Accuracy', 'F1 Score (Weighted)', 'Kappa Score', 'MCC Score',
        'CV Mean Accuracy', 'CV Mean F1'
    ],
    'Value': [
        round(map50, 4),     round(map5095, 4),
        round(precision, 4), round(recall, 4),
        round(accuracy, 4),  round(f1, 4),
        round(kappa, 4),     round(mcc, 4),
        round(df_kfold['Accuracy'].mean(), 4),
        round(df_kfold['F1 Score'].mean(), 4)
    ]
}

for cls_name in class_names:
    if cls_name in roc_auc_scores:
        summary['Metric'].append(f'ROC-AUC ({cls_name})')
        summary['Value'].append(round(roc_auc_scores[cls_name], 4))
    if cls_name in f1_scores:
        summary['Metric'].append(f'Best F1 ({cls_name})')
        summary['Value'].append(round(f1_scores[cls_name], 4))

df_summary = pd.DataFrame(summary)
df_summary.to_csv('evaluation_results/metrics_summary.csv',    index=False)

print("CSV  saved!")
print(df_summary.to_string(index=False))

CSV  saved!
               Metric  Value
               mAP@50 0.9574
            mAP@50-95 0.6946
            Precision 0.9418
               Recall 0.9190
             Accuracy 0.8123
  F1 Score (Weighted) 0.8110
          Kappa Score 0.7805
            MCC Score 0.7818
     CV Mean Accuracy 0.9752
           CV Mean F1 0.9035
 ROC-AUC (motorcycle) 0.9945
 Best F1 (motorcycle) 0.9091
     ROC-AUC (person) 0.9879
     Best F1 (person) 0.9600
  ROC-AUC (plant pot) 1.0000
  Best F1 (plant pot) 0.9961
       ROC-AUC (pole) 0.9868
       Best F1 (pole) 0.9497
    ROC-AUC (pothole) 0.9971
    Best F1 (pothole) 0.9901
       ROC-AUC (tree) 0.9990
       Best F1 (tree) 0.9822
ROC-AUC (zebra cross) 0.9997
Best F1 (zebra cross) 0.9933


In [19]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 13)
        self.cell(0, 10, 'Drishti Obstacle Detection - Model Evaluation Report',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

pdf = PDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Title
pdf.set_font('Helvetica', 'B', 18)
pdf.cell(0, 12, 'Model Evaluation Report', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 11)
pdf.cell(0, 8, 'Model: YOLOv11n | Drishti Obstacle Detection System',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)

# Metrics Table
pdf.set_font('Helvetica', 'B', 13)
pdf.cell(0, 10, '1. Metrics Summary', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', 'B', 10)
pdf.cell(100, 8, 'Metric', border=1)
pdf.cell(80,  8, 'Value',  border=1, new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
for _, row in df_summary.iterrows():
    pdf.cell(100, 8, str(row['Metric']), border=1)
    pdf.cell(80,  8, str(row['Value']),  border=1, new_x='LMARGIN', new_y='NEXT')

# Overfitting section
pdf.ln(5)
pdf.set_font('Helvetica', 'B', 13)
pdf.cell(0, 10, '2. Overfitting Analysis', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 8, f'CV Mean Accuracy : {df_kfold["Accuracy"].mean():.4f}',
         new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 8, f'Test Accuracy    : {accuracy:.4f}',
         new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 8, f'Difference       : {diff:.4f} - {"No overfitting detected" if diff < 0.05 else "Slight overfitting"}',
         new_x='LMARGIN', new_y='NEXT')

# Add all chart images
images = [
    ('3. K-Fold Cross Validation',  'evaluation_results/kfold_crossvalidation.png'),
    ('4. Confusion Matrix',          'evaluation_results/confusion_matrix.png'),
    ('5. ROC-AUC Curve',            'evaluation_results/roc_auc_curve.png'),
    ('6. Precision-Recall Curve',   'evaluation_results/pr_curve.png'),
    ('7. F1-Confidence Curve',      'evaluation_results/f1_confidence_curve.png'),
    ('8. Overfitting Check',        'evaluation_results/overfitting_check.png'),
]

for title, img_path in images:
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.set_font('Helvetica', 'B', 13)
        pdf.cell(0, 10, title, new_x='LMARGIN', new_y='NEXT')
        pdf.image(img_path, x=10, w=190)

pdf.output('evaluation_results/evaluation_report.pdf')

print("\nALL FILES SAVED in evaluation_results/")
print("   metrics_summary.csv")
print("   metrics_summary.xlsx")
print("   confusion_matrix.png")
print("   roc_auc_curve.png")
print("   pr_curve.png")
print("   f1_confidence_curve.png")
print("   kfold_crossvalidation.png")
print("   overfitting_check.png")
print("   evaluation_report.pdf")


ALL FILES SAVED in evaluation_results/
   metrics_summary.csv
   metrics_summary.xlsx
   confusion_matrix.png
   roc_auc_curve.png
   pr_curve.png
   f1_confidence_curve.png
   kfold_crossvalidation.png
   overfitting_check.png
   evaluation_report.pdf
